In [1]:
# ============================================
# CELL 1: SETUP & IMPORTS
# ============================================
!pip install timm einops -q

import os, json, math
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torchvision.transforms as transforms
from PIL import Image

from timm.models.vision_transformer import Block, PatchEmbed
from einops import rearrange
import matplotlib.pyplot as plt

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = True

set_seed(42)
DEBUG = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory: {total_mem:.2f} GB")
print("\nSetup complete!")


Device: cuda
GPU: Tesla T4
Memory: 15.64 GB

Setup complete!


In [2]:
# ============================================
# CELL 2: DATASET & DATALOADER
# ============================================

class CelebAMAEDataset(Dataset):
    def __init__(self, data_dir, mask_pt_path=None, split='train', train_ratio=0.9):
        base_dir = Path(data_dir)
        possible_paths = [
            base_dir / "data" / "processed",
            base_dir / "processed",
            base_dir
        ]
        self.data_dir = None
        for path in possible_paths:
            if (path / "images").exists():
                self.data_dir = path
                break
        if self.data_dir is None:
            raise FileNotFoundError("Cannot find images folder")

        self.image_paths = sorted((self.data_dir / "images").glob("*.jpg"))
        split_idx = int(len(self.image_paths) * train_ratio)
        self.image_paths = self.image_paths[:split_idx] if split == 'train' else self.image_paths[split_idx:]

        if split == 'train':
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])

        # FIX TẠI ĐÂY: Load toàn bộ mask từ file gộp .pt lên RAM
        self.all_masks = {}
        self.has_region_masks = False
        if mask_pt_path and Path(mask_pt_path).exists():
            print(f"Loading cached masks into RAM from {mask_pt_path}...")
            self.all_masks = torch.load(mask_pt_path)
            self.has_region_masks = True
        else:
            print("Warning: Không tìm thấy file mask_pt_path, sẽ dùng mask toàn số 0.")

        print(f" {split} dataset: {len(self.image_paths)} samples | region_masks loaded: {self.has_region_masks}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img_id   = img_path.stem
        img      = Image.open(img_path).convert('RGB')
        img      = self.transform(img)

        # Lấy mask trực tiếp từ RAM (Dictionary map), tốc độ O(1), không tốn disk I/O
        region_mask = torch.zeros(196, dtype=torch.long)
        if self.has_region_masks and img_id in self.all_masks:
            region_mask = self.all_masks[img_id]
            
        return img, region_mask

# Sửa hàm get_dataloaders để nhận thêm đường dẫn file mask gộp
def get_dataloaders(data_dir, mask_pt_path, batch_size=64, num_workers=4):
    train_dataset = CelebAMAEDataset(data_dir, mask_pt_path=mask_pt_path, split='train')
    val_dataset   = CelebAMAEDataset(data_dir, mask_pt_path=mask_pt_path, split='val')

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, num_workers=num_workers,
                              pin_memory=True, drop_last=True,
                              persistent_workers=True, prefetch_factor=2)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size * 2,
                              shuffle=False, num_workers=num_workers,
                              pin_memory=True, persistent_workers=True,
                              prefetch_factor=2)

    return train_loader, val_loader


DATA_DIR    = "/kaggle/input/datasets/doandangkhoa/celeba-processed-224"
MASK_PT_PATH = "/kaggle/input/datasets/doandangkhoa/mae-compare/all_region_masks.pt"
BATCH_SIZE  = 256
NUM_WORKERS = 4

train_loader, val_loader = get_dataloaders(DATA_DIR, MASK_PT_PATH, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
print("\n Dataloaders ready!")


Loading cached masks into RAM from /kaggle/input/datasets/doandangkhoa/mae-compare/all_region_masks.pt...
 train dataset: 182302 samples | region_masks loaded: True
Loading cached masks into RAM from /kaggle/input/datasets/doandangkhoa/mae-compare/all_region_masks.pt...
 val dataset: 20256 samples | region_masks loaded: True

 Dataloaders ready!


In [4]:
# ============================================
# CELL 3: MASKING STRATEGIES
# ============================================

class MaskingStrategy:
    def __init__(self, mask_ratio=0.75):
        self.mask_ratio  = mask_ratio
        self.num_patches = 196

    def generate_mask(self, batch_size, region_masks=None):
        raise NotImplementedError


class RandomMasking(MaskingStrategy):
    def generate_mask(self, batch_size, region_masks=None):
        num_masked  = int(self.mask_ratio * self.num_patches)
        noise       = torch.rand(batch_size, self.num_patches)
        ids_shuffle = torch.argsort(noise, dim=1)
        mask        = torch.zeros(batch_size, self.num_patches, dtype=torch.bool)
        mask.scatter_(1, ids_shuffle[:, :num_masked], True)
        return mask

    def generate_mask(self, batch_size, region_masks=None):
        num_blocks_per_side = self.grid_size // self.block_size
        num_blocks_total    = num_blocks_per_side ** 2
        num_blocks_to_mask  = int(self.mask_ratio * num_blocks_total)
        masks = []
        for _ in range(batch_size):
            block_indices = torch.randperm(num_blocks_total)[:num_blocks_to_mask]
            mask = torch.zeros(self.grid_size, self.grid_size, dtype=torch.bool)
            for block_idx in block_indices:
                bi = block_idx // num_blocks_per_side
                bj = block_idx  % num_blocks_per_side
                si, sj = bi * self.block_size, bj * self.block_size
                mask[si:si+self.block_size, sj:sj+self.block_size] = True
            masks.append(mask.flatten())
        return torch.stack(masks)


class FaceAwareMasking(MaskingStrategy):
    """
    mask_ratio     : tỷ lệ mask tổng thể (cố định 0.75)
    critical_ratio : trong K critical patches, bao nhiêu % bị mask
                     — đây là biến thực nghiệm (0.7 / 0.8 / 0.9)
    """
    def __init__(self, mask_ratio=0.75, critical_ratio=0.9):
        super().__init__(mask_ratio)
        self.critical_ratio = critical_ratio

    def generate_mask(self, batch_size, region_masks=None):
        if region_masks is None:
            return RandomMasking(self.mask_ratio).generate_mask(batch_size)

        num_to_mask = int(self.mask_ratio * self.num_patches)
        masks = []
        for i in range(batch_size):
            rm         = region_masks[i].cpu()
            critical   = (rm == 1).nonzero(as_tuple=True)[0]
            background = (rm == 0).nonzero(as_tuple=True)[0]

            num_critical_mask = min(int(self.critical_ratio * len(critical)), num_to_mask)
            crit_perm = critical[torch.randperm(len(critical))[:num_critical_mask]]

            num_bg_mask = num_to_mask - num_critical_mask
            bg_perm     = background[torch.randperm(len(background))[:num_bg_mask]]

            mask = torch.zeros(self.num_patches, dtype=torch.bool)
            mask[crit_perm] = True
            mask[bg_perm]   = True
            masks.append(mask)
        return torch.stack(masks)


print(" Masking strategies ready!")


 Masking strategies ready!


In [5]:
# ============================================
# CELL 4: MAE ENCODER (ViT-Small)
# ============================================

class MAEEncoder(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3,
                 embed_dim=384, depth=6, num_heads=6, mlp_ratio=4.0):
        super().__init__()
        self.patch_size  = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.embed_dim   = embed_dim

        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size,
                                       in_chans=in_chans, embed_dim=embed_dim)
        self.pos_embed   = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        self.blocks      = nn.ModuleList([
            Block(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio,
                  qkv_bias=True, norm_layer=nn.LayerNorm)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        w = self.patch_embed.proj.weight.data
        nn.init.xavier_uniform_(w.view([w.shape[0], -1]))
        self.apply(self._init_block_weights)

    def _init_block_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0); nn.init.constant_(m.weight, 1.0)

    def forward(self, x, mask):
        B = x.shape[0]
        x = self.patch_embed(x) + self.pos_embed

        visible_mask = ~mask
        num_visible  = visible_mask[0].sum().item()
        vis_idx      = visible_mask.float().argsort(dim=1, descending=True)[:, :num_visible]
        x_visible    = torch.gather(x, 1, vis_idx.unsqueeze(-1).expand(-1, -1, x.shape[-1]))

        for block in self.blocks:
            x_visible = block(x_visible)
        x_visible = self.norm(x_visible)
        return x_visible, vis_idx


# Quick test
encoder   = MAEEncoder(embed_dim=384, depth=6, num_heads=6).to(device)
test_imgs = torch.randn(2, 3, 224, 224).to(device)
test_mask = torch.rand(2, 196).to(device) < 0.75
with torch.no_grad():
    enc_out, vis_idx = encoder(test_imgs, test_mask)
print(f" Encoder output: {enc_out.shape}")
print(f"  Params: {sum(p.numel() for p in encoder.parameters())/1e6:.1f}M")
del encoder, test_imgs, test_mask, enc_out, vis_idx
torch.cuda.empty_cache()


 Encoder output: torch.Size([2, 49, 384])
  Params: 11.0M


In [6]:
# ============================================
# CELL 5: MAE DECODER
# ============================================

class MAEDecoder(nn.Module):
    def __init__(self, num_patches=196, patch_size=16, in_chans=3,
                 encoder_embed_dim=384, decoder_embed_dim=256,
                 decoder_depth=4, decoder_num_heads=8, mlp_ratio=4.0):
        super().__init__()
        self.num_patches = num_patches
        self.patch_size  = patch_size
        self.in_chans    = in_chans

        self.decoder_embed     = nn.Linear(encoder_embed_dim, decoder_embed_dim)
        self.mask_token        = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, num_patches, decoder_embed_dim))
        self.decoder_blocks    = nn.ModuleList([
            Block(dim=decoder_embed_dim, num_heads=decoder_num_heads, mlp_ratio=mlp_ratio,
                  qkv_bias=True, norm_layer=nn.LayerNorm)
            for _ in range(decoder_depth)
        ])
        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size ** 2 * in_chans)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        nn.init.trunc_normal_(self.decoder_pos_embed, std=0.02)
        self.apply(self._init_block_weights)

    def _init_block_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0); nn.init.constant_(m.weight, 1.0)

    def forward(self, x_visible, vis_idx, mask):
        B = x_visible.shape[0]
        x_visible = self.decoder_embed(x_visible)
        D = x_visible.shape[-1]

        x_full = self.mask_token.to(x_visible.dtype).expand(B, self.num_patches, -1).clone()
        x_full.scatter_(1, vis_idx.unsqueeze(-1).expand(-1, -1, D), x_visible)
        x_full = x_full + self.decoder_pos_embed.to(x_visible.dtype)

        for block in self.decoder_blocks:
            x_full = block(x_full)
        x_full = self.decoder_norm(x_full)
        x_full = self.decoder_pred(x_full)
        return x_full


print(" Decoder ready!")


 Decoder ready!


In [ ]:
# ============================================
# CELL 6: COMPLETE MAE MODEL
# ============================================

class MAE(nn.Module):
    def __init__(
        self,
        img_size=224, patch_size=16, in_chans=3,
        encoder_embed_dim=384, encoder_depth=6, encoder_num_heads=6,
        decoder_embed_dim=256, decoder_depth=4, decoder_num_heads=8,
        mlp_ratio=4.0,
        masking_strategy='face_aware',
        mask_ratio=0.75,       
        critical_ratio=0.9,    
        norm_pix_loss=True,
        region_weighted=True,
    ):
        super().__init__()
        self.patch_size      = patch_size
        self.num_patches     = (img_size // patch_size) ** 2
        self.norm_pix_loss   = norm_pix_loss
        self.region_weighted = region_weighted

        self.encoder = MAEEncoder(
            img_size=img_size, patch_size=patch_size, in_chans=in_chans,
            embed_dim=encoder_embed_dim, depth=encoder_depth,
            num_heads=encoder_num_heads, mlp_ratio=mlp_ratio
        )
        self.decoder = MAEDecoder(
            num_patches=self.num_patches, patch_size=patch_size, in_chans=in_chans,
            encoder_embed_dim=encoder_embed_dim,
            decoder_embed_dim=decoder_embed_dim,
            decoder_depth=decoder_depth,
            decoder_num_heads=decoder_num_heads,
            mlp_ratio=mlp_ratio
        )

        if masking_strategy == 'random':
            self.masker = RandomMasking(mask_ratio)
        elif masking_strategy == 'face_aware':
            self.masker = FaceAwareMasking(mask_ratio, critical_ratio=critical_ratio)
        else:
            raise ValueError(f"Unknown masking strategy: {masking_strategy}")

    def patchify(self, imgs):
        p = self.patch_size
        return rearrange(imgs, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=p, p2=p)

    def unpatchify(self, x):
        p   = self.patch_size
        h = w = int(x.shape[1] ** 0.5)
        return rearrange(x, 'b (h w) (p1 p2 c) -> b c (h p1) (w p2)', h=h, w=w, p1=p, p2=p)

    def _compute_loss(self, imgs, pred, mask, region_masks=None):
        target = self.patchify(imgs)
        if self.norm_pix_loss:
            mean   = target.mean(dim=-1, keepdim=True)
            var    = target.var(dim=-1, keepdim=True)
            target = (target - mean) / (var + 1e-6).sqrt()
        loss       = ((pred - target) ** 2).mean(dim=-1)   # [B, 196]
        mask_float = mask.float()
        if self.region_weighted and region_masks is not None:
            weights = 2.0 * region_masks.float() + 0.5 * (1 - region_masks.float())
            loss    = loss * weights
        return (loss * mask_float).sum() / (mask_float.sum() + 1e-6)

    def forward(self, imgs, region_masks=None):
        mask            = self.masker.generate_mask(imgs.shape[0], region_masks).to(imgs.device)
        latent, vis_idx = self.encoder(imgs, mask)
        pred            = self.decoder(latent, vis_idx, mask)
        loss            = self._compute_loss(imgs, pred, mask, region_masks)
        return loss, pred, mask


# ============ Khởi tạo model ============
model = MAE(
    encoder_embed_dim=384, encoder_depth=6,  encoder_num_heads=6,
    decoder_embed_dim=256, decoder_depth=4,  decoder_num_heads=8,
    masking_strategy='face_aware',
    mask_ratio=0.75,
    critical_ratio=0.9,
    norm_pix_loss=True,
    region_weighted=True,
).to(device)

# 1. IN THÔNG TIN VÀ TÍNH THAM SỐ 
print(f" Strategy : {type(model.masker).__name__}")
print(f"Mask_ratio     : {model.masker.mask_ratio}")      
print(f" Critical_ratio : {model.masker.critical_ratio}")  

enc_params   = sum(p.numel() for p in model.encoder.parameters()) / 1e6
dec_params   = sum(p.numel() for p in model.decoder.parameters()) / 1e6
total_params = enc_params + dec_params
print(f" MAE model ready! (critical_ratio=0.9)")
print(f"  Encoder: {enc_params:.1f}M | Decoder: {dec_params:.1f}M | Total: {total_params:.1f}M")

# 2. BỌC MULTI-GPU (DATAPARALLEL) 
if torch.cuda.device_count() > 1:
    print(f"Sử dụng {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

# 3. CHẠY SMOKE TEST
test_imgs = torch.randn(4, 3, 224, 224).to(device)
test_rm   = torch.randint(0, 2, (4, 196)).to(device)
with torch.no_grad():
    loss, pred, mask = model(test_imgs, test_rm)
    
loss = loss.mean() 

print(f"  Smoke test loss: {loss.item():.4f} | pred: {pred.shape}")
del test_imgs, test_rm, loss, pred, mask
torch.cuda.empty_cache()

 Strategy : FaceAwareMasking
Mask_ratio     : 0.75
 Critical_ratio : 0.8
 MAE model ready! (critical_ratio=0.8)
  Encoder: 11.0M | Decoder: 3.5M | Total: 14.5M
Sử dụng 2 GPUs!
  Smoke test loss: 1.9121 | pred: torch.Size([4, 196, 768])


In [10]:
# ============================================
# CELL 7: LEARNING RATE SCHEDULER
# ============================================

def cosine_scheduler(base_value, final_value, epochs, warmup_epochs=0, start_warmup_value=0):
    warmup_schedule = np.array([])
    if warmup_epochs > 0:
        warmup_schedule = np.linspace(start_warmup_value, base_value, warmup_epochs)
    iters    = np.arange(epochs - warmup_epochs)
    schedule = final_value + 0.5 * (base_value - final_value) * (1 + np.cos(np.pi * iters / len(iters)))
    schedule = np.concatenate((warmup_schedule, schedule))
    assert len(schedule) == epochs
    return schedule

def adjust_learning_rate(optimizer, epoch, lr_schedule):
    lr = lr_schedule[epoch]
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    return lr

print(" LR scheduler ready!")


 LR scheduler ready!


In [11]:
# ============================================
# CELL 8: TRAINING & VALIDATION FUNCTIONS
# ============================================

def train_one_epoch(model, dataloader, optimizer, scaler, device, epoch, grad_clip=1.0):
    model.train()
    total_loss, total_samples = 0.0, 0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch:3d} [Train]")
    for images, region_masks in pbar:
        images       = images.to(device, non_blocking=True)
        region_masks = region_masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast("cuda"):
            mae_loss, _, _ = model(images, region_masks)
        mae_loss = mae_loss.mean()
        scaler.scale(mae_loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        bs             = images.shape[0]
        total_loss    += mae_loss.item() * bs
        total_samples += bs
        pbar.set_postfix({'loss': f"{mae_loss.item():.4f}",
                          'avg':  f"{total_loss/total_samples:.4f}"})
    return total_loss / total_samples


@torch.no_grad()
def validate(model, dataloader, device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    for images, region_masks in tqdm(dataloader, desc="[Val]", leave=False):
        images       = images.to(device, non_blocking=True)
        region_masks = region_masks.to(device, non_blocking=True)
        with autocast("cuda"):
            mae_loss, _, _ = model(images, region_masks)
        mae_loss = mae_loss.mean()
        bs             = images.shape[0]
        total_loss    += mae_loss.item() * bs
        total_samples += bs
    return total_loss / total_samples

print(" Training & validation functions ready!")


 Training & validation functions ready!


In [ ]:
# ============================================
# CELL 9: MAIN TRAINING LOOP (FIXED RESUME LR)
# ============================================
import shutil

def safe_save(ckpt, primary_path, backup_path=None):
    primary_path = Path(primary_path)
    tmp_path     = primary_path.with_suffix('.tmp')
    torch.save(ckpt, tmp_path)
    tmp_path.replace(primary_path)
    if backup_path:
        shutil.copy2(primary_path, backup_path)


def train_mae(model, train_loader, val_loader,
              epochs=100, base_lr=1.5e-4, min_lr=1e-6,
              warmup_epochs=10, weight_decay=0.05, grad_clip=1.0,
              save_dir="/kaggle/working", model_name="mae_face_aware",
              resume_from=None, save_every=5, device='cuda'):

    print("="*60)
    print(f"TRAINING: {model_name}")
    print(f"  Epochs: {epochs} | LR: {base_lr:.1e} → {min_lr:.1e} | warmup: {warmup_epochs}")
    print(f"  Save every: {save_every} epochs")
    print("="*60)

    save_dir = Path(save_dir)
    ckpt_dir = save_dir / "checkpoints" / model_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    tmp_dir  = Path("/tmp/mae_checkpoints")
    tmp_dir.mkdir(parents=True, exist_ok=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr,
                                  betas=(0.9, 0.95), weight_decay=weight_decay)
    scaler    = GradScaler("cuda")

    start_epoch   = 0
    best_val_loss = float('inf')
    history       = {'train_loss': [], 'val_loss': [], 'lr': []}

    # ── Build LR schedule (FIXED: continue from current LR) ───────
    if resume_from and Path(resume_from).exists():
        _ckpt = torch.load(resume_from, map_location='cpu', weights_only=False)

        # Load states
        model.load_state_dict(_ckpt['model'])
        optimizer.load_state_dict(_ckpt['optimizer'])
        scaler.load_state_dict(_ckpt['scaler'])
        print(f"[DEBUG] Loaded LR from optimizer: {optimizer.param_groups[0]['lr']:.6e}")

        start_epoch   = _ckpt['epoch'] + 1
        best_val_loss = _ckpt.get('best_val_loss', float('inf'))
        history       = _ckpt.get('history', history)

        # Lấy LR hiện tại từ optimizer 
        current_lr = optimizer.param_groups[0]['lr']

        remaining_epochs = epochs - start_epoch
        if remaining_epochs <= 0:
            print("No remaining epochs. Stop training.")
            return model, history

        lr_schedule = cosine_scheduler(
            base_value=current_lr,
            final_value=min_lr,
            epochs=remaining_epochs,
            warmup_epochs=0
        )

        print(f"Resumed from epoch {start_epoch}, best val: {best_val_loss:.4f}, current lr: {current_lr:.2e}")
    else:
        start_epoch = 0
        lr_schedule = cosine_scheduler(
            base_value=base_lr,
            final_value=min_lr,
            epochs=epochs,
            warmup_epochs=warmup_epochs
        )
    # ─────────────────────────────────────────────────────────────

    for epoch in range(start_epoch, epochs):

        if resume_from and Path(resume_from).exists():
            lr = lr_schedule[epoch - start_epoch]
            if DEBUG and epoch == start_epoch:
                print(f"[DEBUG] schedule_idx = {epoch - start_epoch}")
                print(f"[DEBUG] lr_schedule[0] = {lr_schedule[0]:.6e}")
        else:
            lr = lr_schedule[epoch]

        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        if DEBUG and epoch < start_epoch + 3:
            print(f"[DEBUG] Epoch {epoch} LR: {lr:.6e}")
        
            for i, g in enumerate(optimizer.param_groups):
                print(f"[DEBUG] group {i} lr = {g['lr']:.6e}")

        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, device, epoch, grad_clip)
        val_loss   = validate(model, val_loader, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['lr'].append(lr)

        is_best = val_loss < best_val_loss
        if is_best:
            best_val_loss = val_loss

        print(f"Epoch {epoch:3d} | train: {train_loss:.4f} | val: {val_loss:.4f} | lr: {lr:.2e}"
              + (" ← best" if is_best else ""))

        ckpt = {'epoch': epoch, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                'val_loss': val_loss, 'best_val_loss': best_val_loss, 'history': history}

        safe_save(ckpt, ckpt_dir / "latest.pth", tmp_dir / "latest.pth")
        if is_best:
            safe_save(ckpt, ckpt_dir / "best.pth", tmp_dir / "best.pth")
            print("   Saved best checkpoint")
        if (epoch + 1) % save_every == 0:
            safe_save(ckpt, ckpt_dir / f"epoch_{epoch:03d}.pth",
                      tmp_dir / f"epoch_{epoch:03d}.pth")
            print(f"   Saved epoch checkpoint: epoch_{epoch:03d}.pth")

        torch.cuda.empty_cache()

    print(f"\nTraining complete! Best val loss: {best_val_loss:.4f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.plot(history['train_loss'], label='Train')
    ax1.plot(history['val_loss'],   label='Val')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title('Loss Curves'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(history['lr'], color='orange')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('LR')
    ax2.set_title('LR Schedule'); ax2.set_yscale('log'); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / f"{model_name}_curves.png", dpi=150)
    plt.show()

    return model, history

print("Training loop ready!")

Training loop ready!


In [13]:
# ============================================
# CELL 10: RUN TRAINING  (critical_ratio=0.9)
# ============================================

trained_model, history = train_mae(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=100,
    base_lr=1.5e-4,
    min_lr=1e-6,
    warmup_epochs=10,
    weight_decay=0.05,
    grad_clip=1.0,
    save_dir="/kaggle/working",
    model_name="mae_face_aware_rho8",
    save_every=5,
    resume_from=None,
    # resume_from="/kaggle/working/checkpoints/mae_face_aware_rho8/latest.pth",
    device=device
)

TRAINING: mae_face_aware_rho8
  Epochs: 100 | LR: 1.5e-04 → 1.0e-06 | warmup: 10
  Save every: 5 epochs


Epoch   0 [Train]:   0%|          | 0/712 [00:00<?, ?it/s]

[Val]:   0%|          | 0/40 [00:00<?, ?it/s]

Epoch   0 | train: 0.9811 | val: 0.9812 | lr: 0.00e+00 ← best
   Saved best checkpoint


Epoch   1 [Train]:   0%|          | 0/712 [00:00<?, ?it/s]

[Val]:   0%|          | 0/40 [00:00<?, ?it/s]

Epoch   1 | train: 0.5156 | val: 0.4395 | lr: 1.67e-05 ← best
   Saved best checkpoint


Epoch   2 [Train]:   0%|          | 0/712 [00:00<?, ?it/s]

KeyboardInterrupt: 